In [2]:
#CAPSTONE MASSA MATTEO DAPT0325 Sistema di Early Warning per il Rischio di Contenzioso bancario (banche prese in esame Intesa Sanpaolo Unicredit e Fineco)


import pandas as pd 
import numpy as np
import google_play_scraper as gps
from google_play_scraper import Sort , reviews as google_reviews


In [3]:

banche_google = {

    'Intesa': 'com.latuabancaperandroid', 
    'Unicredit': 'com.unicredit',
    'Credem': 'com.CredemMobile',
    'Credit Agricole': 'it.caitalia.apphub',
    'Fineco': 'com.fineco.it',
    'BBVA': 'com.bbva.italy',
    'Revolut': 'com.revolut.revolut',
    'Hype': 'it.hype.app'
}


In [4]:
dati_raccolti = []
ANNO_TARGET = (2024,2023,2022)
max_reviews = 100000


In [5]:
for banca, app_id in banche_google.items():
     
    recensioni_android, _ = google_reviews( 
        app_id,
        lang='it',
        country='it',
        sort=Sort.NEWEST,
        count=max_reviews
    ) 
    for recensione in recensioni_android:
        data_recensione = recensione['at']
        if data_recensione.year in ANNO_TARGET:
            dati_raccolti.append({
                'banca': banca,
                'data': data_recensione,
                'testo': recensione['content'],
                'valutazione': recensione['score']
            })
        elif data_recensione.year < 2022:
            break
    


In [6]:
df_google = pd.DataFrame(dati_raccolti)


In [7]:
df_google.head()
df_google.info()
df_google.sample(3000)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77863 entries, 0 to 77862
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   banca        77863 non-null  object        
 1   data         77863 non-null  datetime64[ns]
 2   testo        77861 non-null  object        
 3   valutazione  77863 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 2.4+ MB


,banca,data,testo,valutazione
38394,Credit Agricole,2023-03-11 16:27:52,Mai avuto difficoltà nelle operazioni,5
12911,Intesa,2023-01-16 09:38:12,che una app di una banca renda impossibile pag...,1
1965,Intesa,2024-11-09 22:04:45,Dopo aver installato e attivato l'app più di u...,3
53660,BBVA,2024-02-29 14:06:10,il migliore,5
41318,Credit Agricole,2022-03-27 13:31:11,Come si fa ad installare l'applicazione su un ...,2
...,...,...,...,...
57817,Revolut,2024-11-29 00:23:13,Mi piace come sistema e sivuto e pratico,5
4302,Intesa,2024-06-10 12:53:21,si apre dopo un minuto buono l'ultimo aggiorna...,1
49141,Fineco,2022-10-09 11:04:19,La password ad 8 cifre è scarsa il codice disp...,1
23147,Unicredit,2024-05-27 21:44:23,Avete una app che fa schifo nessun aggiornamen...,1


In [11]:
df_google.drop_duplicates()
df_google.dropna(subset=['testo'])
df_google.info()

<class 'pandas.core.frame.DataFrame'>
Index: 77861 entries, 0 to 77862
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   banca        77861 non-null  object        
 1   data         77861 non-null  datetime64[ns]
 2   testo        77861 non-null  object        
 3   valutazione  77861 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 3.0+ MB


In [3]:
USER = "root"
PASSWORD = "Matteo00" # Inserisci la tua password di MySQL
HOST = "localhost"
DATABASE = "progetto_banca" # Il nome dello schema creato su Workbench

In [ ]:
sqlalchemy_url = f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}'
from sqlalchemy import create_engine
engine = create_engine(sqlalchemy_url)
df_google = df_google.drop_duplicates().dropna(subset=['testo'])
df_google.to_sql('recensioni_google', con=engine, if_exists='replace', index=False)
